In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from sklearn import datasets
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.cluster import KMeans
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import plotly.express as px
import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output

# armamento del Data Frame proveniente de scikit-learn

data = datasets.fetch_california_housing()

df = pd.DataFrame(data["data"],columns=data["feature_names"])
df["MedHouseVal"] = data["target"]
df["MedHouseVal"] = df["MedHouseVal"] * 100000

df_model = df.copy() # generando una copia, asi no afectamos al Data Frame original

def remove_outliers(df,columns):
    for c in columns:
        df[c] = df[c].mask(zscore(df[c]).abs() > 3, np.nan)
    
    return df

# removiendo valores atípicos que afecten el modelo
df_model = remove_outliers(df_model, df.columns)
df_model.dropna(inplace=True)

# escalamiento de datos
mm_scaler_features = MinMaxScaler()
features_scaled = mm_scaler_features.fit_transform(df_model[df_model.columns[:-1]])
df_features_scaled = pd.DataFrame(features_scaled, columns=df.columns[:-1])

mm_scaler_objective_var = MinMaxScaler()
objective_var_scaled = mm_scaler_objective_var.fit_transform(df_model["MedHouseVal"].values.reshape(-1, 1))

fig, ax = plt.subplots(2,4, figsize=(21,10))

def figs_scaler(var, index_row, index_col):
    ax[index_row, index_col].scatter(df_features_scaled[var], objective_var_scaled.T)
    ax[index_row, index_col].set_title(var)
    ax[index_row, index_col].set_ylabel("Variable objetivo escalada")
    ax[index_row, index_col].set_xlabel("Variable independiente escalada")
    ax[index_row, index_col].grid("on")

figs_scaler(df.columns[0], 0, 0)
figs_scaler(df.columns[1], 0, 1)
figs_scaler(df.columns[2], 0, 2)
figs_scaler(df.columns[3], 0, 3)
figs_scaler(df.columns[4], 1, 0)
figs_scaler(df.columns[5], 1, 1)
figs_scaler(df.columns[6], 1, 2)
figs_scaler(df.columns[7], 1, 3)

plt.subplots_adjust(wspace=0.3, hspace=0.3)
plt.show()

#### Random Search CV
Es una función en la que se crean modelos con los diferentes parámetros otorgados para luego elegir aleatoriamente algunos de ellos, realizar validaciones cruzadas(X cantidad de pruebas con distintos datos de entrenamiento) y obtener un promedio del rendimiento(R2_score) de cada uno.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(features_scaled,
                                                    objective_var_scaled.T,
                                                    test_size=0.25)

# ajuste de hiperparámetros utilizando Random Search 

xgbr_test = XGBRegressor()           

turned_parameters = {
    "n_estimators":[100,200,300,400],
    "subsample":[0.7,0.75,0.8,0.85],
    "max_depth":[3,4,5,6],
    "learning_rate":[0.2,0.3,0.4,0.5],
    "min_child_weight":[2,3,4,5],
    "gamma":[0,1,2,3]
}

grid_search = RandomizedSearchCV(xgbr_test, turned_parameters,cv=5)
grid_search.fit(x_train, y_train)

print(f'''n_estimators(número de modelos creados): {grid_search.best_params_["n_estimators"]}
          subsample(tamaño de muestra requerida): {grid_search.best_params_["subsample"]}
          max_depth(máxima profundidad de cada árbol): {grid_search.best_params_["max_depth"]}
          learning_rate(tasa de aprendizaje, evita el sobreajuste): {grid_search.best_params_["learning_rate"]}
          min_child_weight(suma mínima de peso de instancia necesaria en un nodo): {grid_search.best_params_["min_child_weight"]}
          gamma(cuanto mayor sea, más conservador será el modelo): {grid_search.best_params_["gamma"]}''')

#### Boosting
En el caso anterior habiamos mencionado y empleado la técnica de Bagging, donde generábamos distintos modelos entrenados de forma independiente. En este metaaprendizaje cada modelo capacitará y solucionará los errores del siguiente potenciando cada vez más la precisión final, es decir que su escalabilidad es vertical. Requieren de una hiperparametrización compleja y un cuidado a la hora del sobreajuste, es decir, modelos con una precisión particular sobre los datos que utilizó en su entrenamiento pero con la poca adaptación para resolver nuevas situaciones.

In [ ]:
# asignándole al modelo los parámetros que obtuvieron mejores resultados

xgbr = XGBRegressor(n_estimators = grid_search.best_params_["n_estimators"],
                    subsample = grid_search.best_params_["subsample"],
                    max_depth = grid_search.best_params_["max_depth"],
                    learning_rate = grid_search.best_params_["learning_rate"],
                    min_child_weight = grid_search.best_params_["min_child_weight"],
                    gamma = grid_search.best_params_["gamma"])

model = xgbr.fit(x_train, y_train)

# predicción de variables y conversión a su formato
predicts = mm_scaler_objective_var.inverse_transform(model.predict(x_test).reshape(-1, 1))
y_test = mm_scaler_objective_var.inverse_transform(y_test.reshape(-1, 1))

# validación con métricas de rendimiento
print(f"MAE: {round(mean_absolute_error(predicts, y_test))} | MSE: {round(mean_squared_error(predicts, y_test))} | RMSE: {round(mean_squared_error(predicts, y_test, squared=True))}")
print("---------------------------------------------------------------------------------")

### Métricas de rendimiento para modelos de regresión
Tienen mucha similitud con las medidas de variabilidad en cuanto a las fórmulas estadísticas que emplean, solo que utilizando los residuos, es decir, las diferencias entre las predicciones del modelo regresor y los valores reales, un modelo óptimo busca la menor variabilidad entre estos componentes. Aglunos ejemplos de estas métricas son:

Error absoluto medio: similar a la desviacíon media absoluta, es la división por el número de muestra de la sumatoria de los residuos absolutos.

Error cuadrático medio: similar a la varianza, es la división por el número de muestra de la sumatoria de los residuos al cuadrado.

Raíz cuadrada del error cuadrático medio: similar a la desviación éstandar, se utiliza para convertir a la unidad en la que se basan los valores.

Coeficiente de determinación(R2 Score): valor de 0 a 1 que evalúa la bondad de ajuste de nuestro modelo a los datos, pero solo utilizada en modelos lineales.
____________________________________________________________________________________________________________________________________________________________________________

#### Dashboard que funciona como mapa descriptivo de las viviendas de California

In [ ]:
app = dash.Dash(__name__)

app.layout = html.Div(id="body",className="e5_body",children=[
    html.H1("Viviendas en California ",id="title",className="e5_title"),
        html.Div(id="div",className="e5_div",children=[
            dcc.Dropdown(id="dropdown",className="e5_dropdown",
                        options = [
                            {"label":"Ingreso medio","value":"MedInc"},
                            {"label":"Edad media","value":"HouseAge"},
                            {"label":"Promedio de habitaciones","value":"AveRooms"},
                            {"label":"Promedio de dormitorios","value":"AveBedrms"},
                            {"label":"Población","value":"Population"},
                            {"label":"Promedio de ocupación","value":"AveOccuption"}
                        ],
                        value="MedInc",
                        multi=False,
                        clearable=False)]),
            dcc.Graph(id="graph",className="e5_graph",figure={})
])

@app.callback(
    Output(component_id="graph",component_property="figure"),
    [Input(component_id="dropdown",component_property="value")]
)

def update_graph(slct_var):
    
    scatter_map = px.scatter(df,x="Longitude",y="Latitude",color=slct_var)
    
    return scatter_map
    
if __name__ == "__main__":
    app.run_server(debug=False)

### Aprendizaje no supervisado
las observaciones no tienen una respuesta asociada que guíe el aprendizaje, uno de sus algoritmos es Kmeans y genera. mediante un proceso iterativo, sus propias etiquetas determinando grupos de datos asociables en función de sus acercamientos estadísticos.

In [ ]:
clusters = []
inertias = []

for c in range(3,12):
    kmeans = KMeans(n_clusters=c).fit(df["MedHouseVal"].values.reshape((-1,1)))
    clusters.append(c)
    inertias.append(kmeans.inertia_)
    
kmeans = KMeans(n_clusters=5).fit(df["MedHouseVal"].values.reshape((-1,1)))
inertia = kmeans.inertia_

plt.plot(clusters, inertias, marker="o")
plt.text(int(str(kmeans)[-2]) - 0.1, inertia, "Valor del codo")
plt.grid("on")
plt.show()

#### Método del codo
El modelo de clustering requiere de un hiperparámetro que es la cantidad de centroides o K-means, estos serán dispersados e irán apropiandose de los datos más cercanos hasta convertirse en las medias de los grupos que formaron(nuevamente, mediante pruebas iterativas), el método del codo se utiliza a la hora de designar este valor, dónde el objetivo es visualizar la menor cantidad de centroides y la menor inercia(alejamiento entre los miembros de clusters y su centroide). También existen métricas especializadas en evaluar esta clase de modelos, Coeficiente de silueta e Índice Davies-Bouldin.

In [ ]:
kmeans = KMeans(n_clusters=5).fit(df["MedHouseVal"].values.reshape((-1,1)))

clusters = kmeans.labels_

df["clusters"] = clusters

range_values = np.array([])

for c in df["clusters"].sort_values().unique():
    cluster = df.loc[df["clusters"] == c,["clusters","MedHouseVal"]]
    max_value = str(cluster["MedHouseVal"].max())
    min_value = str(cluster["MedHouseVal"].min())
    range_values = np.append(range_values,min_value)
    range_values = np.append(range_values,max_value)
    
range_values = range_values.reshape((-1,2))
    
df["clusters"] = df["clusters"].replace(
    {
        0:f"0 ({range_values[0,0][:8]}$-{range_values[0,1][:8]}$)",
        1:f"1 ({range_values[1,0][:8]}$-{range_values[1,1][:8]}$)",
        2:f"2 ({range_values[2,0][:8]}$-{range_values[2,1][:8]}$)",
        3:f"3 ({range_values[3,0][:8]}$-{range_values[3,1][:8]}$)",
        4:f"4 ({range_values[4,0][:8]}$-{range_values[4,1][:8]}$)"
    })


fig = px.scatter(df,x="Longitude",y="Latitude",color="clusters")
fig.update_layout(title="Range of Houses'values")
fig 

##### Grupos de precios de las viviendas
Ya tenemos una idea clara en cuanto a los rangos de precios que las viviendas suelen frecuentar y como su ubicación influye, por ejemplo: vemos un evindente aumento en las viviendas más cercanas a la costa de California, esto ubicando los cluster donde dentro de estos se encuentran los valores más caros siendo (296.800-414.100) y (414.300-500.000), también, observamos que los precios suelen valer entre 15.000 y 127.800 gracias al cluster que guarda esos valores y siendo el más numeroso